# Causal Trace: Canonical

Runs token-by-layer canonical causal tracing and displays the generated heatmap.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import hydra
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from src.causal_trace.prototype import run_canonical_trace

MODEL = 'gpt2-large'
NUM_PROMPTS = 1
NUM_NOISE = 10
POSITION_SCOPE = 'all_tokens'
COMPONENT = 'residual'


In [ ]:
with hydra.initialize_config_dir(config_dir=str(ROOT / 'src' / 'config'), version_base=None):
    cfg = hydra.compose(
        config_name='latium',
        overrides=[
            'command=canonical_trace',
            f'model={MODEL}',
            f'generation.num_of_runs={NUM_PROMPTS}',
            f'tracing.num_noise_samples={NUM_NOISE}',
            f'tracing.position_scope={POSITION_SCOPE}',
            f'tracing.component={COMPONENT}',
        ],
    )

out_dir = Path(run_canonical_trace(cfg))
long = pd.read_csv(out_dir / 'canonical_trace_long.csv')
long.head()


In [ ]:
matrix = long.pivot(index='position', columns='layer', values='mean_indirect_effect').sort_index()
fig, ax = plt.subplots(figsize=(12, max(4, len(matrix) * 0.35)))
sns.heatmap(matrix, ax=ax, cmap='viridis')
ax.set_title('Canonical token-by-layer indirect effect')
plt.show()

pd.read_json(out_dir / 'canonical_trace_summary.json', typ='series')
